In [ ]:
from selenium import webdriver
from selenium . webdriver.common.by import By
from selenium.webdriver.firefox.service import Service as FirefoxService
from webdriver_manager.firefox import GeckoDriverManager

from selenium.webdriver.chrome.options import Options


opts = Options()
opts.add_argument("--headless")          
opts.add_argument("--no-sandbox")        
opts.add_argument("--disable-dev-shm-usage")


In [ ]:
driver = webdriver.Firefox(service=FirefoxService(GeckoDriverManager().install()))
driver.get("https://fbref.com/en/comps/9/Premier-League-Stats")

In [ ]:
players = []

content = driver.find_element(By.ID, "div_results2025-202691_overall")

In [ ]:
buttons = driver.find_elements(By.XPATH, ".//td[@data-stat='team']/a")

In [ ]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 10)
league_url = driver.current_url

buttons = driver.find_elements(By.XPATH, "//td[@data-stat='team']/a")
hrefs = [a.get_attribute('href') for a in buttons]

for href in hrefs:
    driver.get(href)
    table = wait.until(EC.presence_of_element_located((By.ID, "all_stats_standard")))
    #result.append(table.get_attribute('outerHTML'))
    results = table.find_elements(By.CSS_SELECTOR, '[data-stat="player"]')
    results = table.find_elements(By.CSS_SELECTOR, '[data-stat="player"]')
    results = table.find_elements(By.CSS_SELECTOR , '[data-stat="player"]')
    for result in results:
     
        players.append(result.text)
    


In [51]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.service import Service as FirefoxService
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.firefox import GeckoDriverManager
import pandas as pd

driver = webdriver.Firefox(service=FirefoxService(GeckoDriverManager().install()))
wait = WebDriverWait(driver, 10)

try:
    driver.get("https://fbref.com/en/comps/9/2024-2025/2024-2025-Premier-League-Stats")
    team_links = [a.get_attribute("href") for a in driver.find_elements(By.XPATH, "//td[@data-stat='team']/a")][:20]
    teams = [a.text for a in driver.find_elements(By.XPATH, "//td[@data-stat='team']/a")][:20]
    
    all_rows = []
    headers = []

    all_rows_match = []
    headers_match = []
    
    for team_idx, link in enumerate(team_links):
        driver.get(link)
        table = wait.until(EC.presence_of_element_located((By.ID, "stats_standard_9")))
        
        if not headers:
            header_row = table.find_element(By.XPATH, ".//thead//tr[last()]")
            all_headers = [th.get_attribute("data-stat") or th.text.strip() for th in header_row.find_elements(By.TAG_NAME, "th")]
            headers = all_headers[:16]
        
        rows = table.find_elements(By.XPATH, ".//tbody//tr[not(contains(@class, 'thead'))]")
        
        current_team = teams[team_idx]
        
        for row in rows:
            row_data = {}
            cells = row.find_elements(By.TAG_NAME, "th") + row.find_elements(By.TAG_NAME, "td")
            for cell_idx, cell in enumerate(cells[:16]):
                if cell_idx < len(headers):
                    data_stat = cell.get_attribute("data-stat")
                    if data_stat and data_stat not in row_data:
                        row_data[data_stat] = cell.text.strip()
            
            if row_data:
                row_data['team'] = current_team
                all_rows.append(row_data)


        table_match = wait.until(EC.presence_of_element_located((By.ID , 'matchlogs_for')))
        header_row_match = table_match.find_element(By.XPATH, ".//thead//tr[last()]")
        headers_match = [th.get_attribute("data-stat") or th.text.strip() 
                     for th in header_row_match.find_elements(By.TAG_NAME, "th")]

        rows_match = table_match.find_elements(By.XPATH, ".//tbody//tr[not(contains(@class, 'thead'))]")

        for row in rows_match:
            row_data = {}
            cells = row.find_elements(By.TAG_NAME, "th") + row.find_elements(By.TAG_NAME, "td")
            for cell_idx, cell in enumerate(cells):
                if cell_idx < len(headers_match):
                    data_stat = cell.get_attribute("data-stat")
                    if data_stat and data_stat not in row_data:
                        row_data[data_stat] = cell.text.strip()
            
            if row_data:
                row_data['team'] = current_team
                all_rows_match.append(row_data)



    
    df = pd.DataFrame(all_rows)
    column_order = headers + ['team']
    df = df[column_order]
    df.to_csv("premier_league_stats.csv", index=False)

    df_match = pd.DataFrame(all_rows_match)

    column_order_match = headers_match + ['team']
    df_match = df_match[column_order_match]
       

   

except Exception as e:
    print(f"Error: {e}")
finally:
    driver.quit()

In [52]:
df_match = df_match.drop(columns=['match_report', 'notes'])


In [ ]:

condition_to_remove = (df_match['venue'] == 'Away') & (df_match['comp'] == 'Premier League')


df_match = df_match[~condition_to_remove]

df_match.to_csv("premier_league_matchs.csv" , index=False) 

In [55]:
df_match[df_match['team'] == 'Liverpool']

,date,start_time,comp,round,dayofweek,venue,result,goals_for,goals_against,opponent,xg_for,xg_against,possession,attendance,captain,formation,opp_formation,referee,team
1,2024-08-25,16:30,Premier League,Matchweek 2,Sun,Home,W,2,0,Brentford,2.5,0.5,62,"60,017",Virgil van Dijk,4-2-3-1,4-4-2,Stuart Attwell,Liverpool
3,2024-09-14,15:00,Premier League,Matchweek 4,Sat,Home,L,0,1,Nott'ham Forest,0.9,0.4,68,"60,344",Virgil van Dijk,4-2-3-1,4-2-3-1,Michael Oliver,Liverpool
5,2024-09-21,15:00,Premier League,Matchweek 5,Sat,Home,W,3,0,Bournemouth,2.0,1.1,58,"60,347",Virgil van Dijk,4-2-3-1,4-2-3-1,Tony Harrington,Liverpool
6,2024-09-25,20:00,EFL Cup,Third round,Wed,Home,W,5,1,West Ham,,,61,"60,044",Joe Gomez,4-2-3-1,4-2-3-1,Andy Madley,Liverpool
8,2024-10-02,20:00,Champions Lg,League phase,Wed,Home,W,2,0,it Bologna,1.2,0.6,51,"59,816",Virgil van Dijk,4-2-3-1,4-1-4-1,Nikola Dabanović,Liverpool
10,2024-10-20,16:30,Premier League,Matchweek 8,Sun,Home,W,2,1,Chelsea,1.9,1.0,43,"60,277",Virgil van Dijk,4-2-3-1,4-2-3-1,John Brooks,Liverpool
14,2024-11-02,15:00 (16:00),Premier League,Matchweek 10,Sat,Home,W,2,1,Brighton,1.6,1.0,49,"60,331",Virgil van Dijk,4-2-3-1,4-4-2,Tony Harrington,Liverpool
15,2024-11-05,20:00 (21:00),Champions Lg,League phase,Tue,Home,W,4,0,de Leverkusen,3.7,0.9,47,"59,790",Virgil van Dijk,4-3-3,3-5-2,Danny Makkelie,Liverpool
16,2024-11-09,20:00 (21:00),Premier League,Matchweek 11,Sat,Home,W,2,0,Aston Villa,2.0,1.2,62,"60,292",Virgil van Dijk,4-2-3-1,4-2-3-1,David Coote,Liverpool
18,2024-11-27,20:00 (21:00),Champions Lg,League phase,Wed,Home,W,2,0,es Real Madrid,2.7,1.2,63,"59,546",Virgil van Dijk,4-3-3,4-2-2-2,François Letexier,Liverpool


In [ ]:
import numpy as np
df.replace('', np.nan, inplace=True)


In [21]:
import pandas as pd
df_match = pd.read_csv('premier_league_matchs.csv')
# df_match[df_match.isna().any(axis=1)]
df_match.isna().sum()

date              0
start_time        0
comp              0
round             0
dayofweek         0
venue             0
result            0
goals_for         0
goals_against     0
opponent          0
xg_for           65
xg_against       65
possession       16
attendance        3
captain           0
formation         0
opp_formation     0
referee           2
team              0
dtype: int64

In [24]:

df_match['referee'] = df_match['referee'].fillna('unknown')



df_match.fillna(0 , inplace=True)


df_match.isna().sum()

date             0
start_time       0
comp             0
round            0
dayofweek        0
venue            0
result           0
goals_for        0
goals_against    0
opponent         0
xg_for           0
xg_against       0
possession       0
attendance       0
captain          0
formation        0
opp_formation    0
referee          0
team             0
dtype: int64

In [25]:

df_match.to_csv('finale_data_mathcs.csv')
df_match


,date,start_time,comp,round,dayofweek,venue,result,goals_for,goals_against,opponent,xg_for,xg_against,possession,attendance,captain,formation,opp_formation,referee,team
0,2024-08-25,16:30,Premier League,Matchweek 2,Sun,Home,W,2,0,Brentford,2.5,0.5,62.0,"60,017",Virgil van Dijk,4-2-3-1,4-4-2,Stuart Attwell,Liverpool
1,2024-09-14,15:00,Premier League,Matchweek 4,Sat,Home,L,0,1,Nott'ham Forest,0.9,0.4,68.0,"60,344",Virgil van Dijk,4-2-3-1,4-2-3-1,Michael Oliver,Liverpool
2,2024-09-21,15:00,Premier League,Matchweek 5,Sat,Home,W,3,0,Bournemouth,2.0,1.1,58.0,"60,347",Virgil van Dijk,4-2-3-1,4-2-3-1,Tony Harrington,Liverpool
3,2024-09-25,20:00,EFL Cup,Third round,Wed,Home,W,5,1,West Ham,0.0,0.0,61.0,"60,044",Joe Gomez,4-2-3-1,4-2-3-1,Andy Madley,Liverpool
4,2024-10-02,20:00,Champions Lg,League phase,Wed,Home,W,2,0,it Bologna,1.2,0.6,51.0,"59,816",Virgil van Dijk,4-2-3-1,4-1-4-1,Nikola Dabanović,Liverpool
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,2025-04-02,19:45 (18:45),Premier League,Matchweek 30,Wed,Home,D,1,1,Crystal Palace,0.7,0.7,43.0,"30,158",Jack Stephens,3-4-3,3-4-3,Andy Madley,Southampton
484,2025-04-12,15:00,Premier League,Matchweek 32,Sat,Home,L,0,3,Aston Villa,0.3,3.0,40.0,"30,199",Jack Stephens,3-4-3,4-2-3-1,Thomas Bramall,Southampton
485,2025-04-26,15:00,Premier League,Matchweek 34,Sat,Home,L,1,2,Fulham,0.6,2.4,35.0,"28,946",Jack Stephens,3-4-3,4-3-3,Tony Harrington,Southampton
486,2025-05-10,15:00,Premier League,Matchweek 36,Sat,Home,D,0,0,Manchester City,0.1,1.7,28.0,"30,937",Jack Stephens,3-4-3,4-2-3-1,Tim Robinson,Southampton


In [23]:
df_match

,date,start_time,comp,round,dayofweek,venue,result,goals_for,goals_against,opponent,xg_for,xg_against,possession,attendance,captain,formation,opp_formation,referee,team
0,2024-08-25,16:30,Premier League,Matchweek 2,Sun,Home,W,2,0,Brentford,2.5,0.5,62.0,"60,017",Virgil van Dijk,4-2-3-1,4-4-2,Stuart Attwell,Liverpool
1,2024-09-14,15:00,Premier League,Matchweek 4,Sat,Home,L,0,1,Nott'ham Forest,0.9,0.4,68.0,"60,344",Virgil van Dijk,4-2-3-1,4-2-3-1,Michael Oliver,Liverpool
2,2024-09-21,15:00,Premier League,Matchweek 5,Sat,Home,W,3,0,Bournemouth,2.0,1.1,58.0,"60,347",Virgil van Dijk,4-2-3-1,4-2-3-1,Tony Harrington,Liverpool
3,2024-09-25,20:00,EFL Cup,Third round,Wed,Home,W,5,1,West Ham,NaN,NaN,61.0,"60,044",Joe Gomez,4-2-3-1,4-2-3-1,Andy Madley,Liverpool
4,2024-10-02,20:00,Champions Lg,League phase,Wed,Home,W,2,0,it Bologna,1.2,0.6,51.0,"59,816",Virgil van Dijk,4-2-3-1,4-1-4-1,Nikola Dabanović,Liverpool
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,2025-04-02,19:45 (18:45),Premier League,Matchweek 30,Wed,Home,D,1,1,Crystal Palace,0.7,0.7,43.0,"30,158",Jack Stephens,3-4-3,3-4-3,Andy Madley,Southampton
484,2025-04-12,15:00,Premier League,Matchweek 32,Sat,Home,L,0,3,Aston Villa,0.3,3.0,40.0,"30,199",Jack Stephens,3-4-3,4-2-3-1,Thomas Bramall,Southampton
485,2025-04-26,15:00,Premier League,Matchweek 34,Sat,Home,L,1,2,Fulham,0.6,2.4,35.0,"28,946",Jack Stephens,3-4-3,4-3-3,Tony Harrington,Southampton
486,2025-05-10,15:00,Premier League,Matchweek 36,Sat,Home,D,0,0,Manchester City,0.1,1.7,28.0,"30,937",Jack Stephens,3-4-3,4-2-3-1,Tim Robinson,Southampton
